# 04 - Model Diagnostics & Ranking Analysis

This notebook requires a trained model. We load the V3 GBDT model, predict on a validation split, and diagnose: where does the model succeed, where does it fail, and what systematic patterns explain the failures?

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import lightgbm as lgb

from src.data_loader import load_train, make_target, split_val, get_feature_columns
from src.features import build_features, compute_position_propensity, compute_sample_weights
from src.evaluate import evaluate_ndcg, ndcg_at_k

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100
SEED = 42

In [ ]:
# Load data and model
print("Loading training data...")
train_raw = load_train()
train_raw = make_target(train_raw)
train_raw = train_raw.sort_values("srch_id").reset_index(drop=True)

print("Splitting train/val...")
train_split, val_split = split_val(train_raw, val_frac=0.1)
train_split = train_split.sort_values("srch_id").reset_index(drop=True)
val_split = val_split.sort_values("srch_id").reset_index(drop=True)

print("Building features...")
train_feat = build_features(train_split, agg_source=train_split, is_train=True)
val_feat = build_features(val_split, agg_source=train_split, is_train=False)
feature_cols = get_feature_columns(train_feat)
feature_cols = [c for c in feature_cols if c in val_feat.columns]

print(f"Val: {len(val_feat):,} rows, {val_feat['srch_id'].nunique():,} searches, {len(feature_cols)} features")

# Load model
model = lgb.Booster(model_file="models/v3_gbdt.txt") if __import__('os').path.exists("models/v3_gbdt.txt") else None

# If no saved model, retrain quickly
if model is None:
    print("No saved model found, training fresh...")
    propensity = compute_position_propensity(train_raw)
    weights = compute_sample_weights(train_split, propensity)
    train_groups = train_feat.groupby("srch_id").size().values
    val_groups = val_feat.groupby("srch_id").size().values
    dtrain = lgb.Dataset(train_feat[feature_cols], label=train_feat["relevance"],
                         group=train_groups, weight=weights)
    dval = lgb.Dataset(val_feat[feature_cols], label=val_feat["relevance"],
                       group=val_groups, reference=dtrain)
    model = lgb.train(
        {"objective": "lambdarank", "metric": "ndcg", "eval_at": [5],
         "learning_rate": 0.03, "num_leaves": 400, "min_child_samples": 50,
         "subsample": 0.7, "colsample_bytree": 0.6, "reg_alpha": 0.1, "reg_lambda": 1.0,
         "seed": SEED, "verbose": -1, "n_jobs": -1},
        dtrain, num_boost_round=3000, valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)]
    )

val_feat = val_feat.copy()
val_feat["pred_score"] = model.predict(val_feat[feature_cols])

# Compute within-query rank
val_feat["pred_rank"] = val_feat.groupby("srch_id")["pred_score"].rank(ascending=False, method="first").astype(int)
group_sizes = val_feat.groupby("srch_id")["pred_score"].transform("count")
val_feat["pred_rank_pct"] = val_feat["pred_rank"] / group_sizes

overall_ndcg = evaluate_ndcg(val_feat, score_col="pred_score", k=5)
print(f"\nOverall Validation NDCG@5: {overall_ndcg:.5f}")

## 4.1 Overall Model Performance

In [ ]:
# Recall@K and rank statistics for booked hotels
booked = val_feat[val_feat["booking_bool"] == 1].copy()
n_booked_searches = booked["srch_id"].nunique()

# For searches with bookings, what rank does the booked hotel get?
print(f"Searches with bookings in val: {n_booked_searches:,}")
print(f"Total val searches: {val_feat['srch_id'].nunique():,}")
print()

recall_at = {}
for k in [1, 3, 5, 10]:
    in_top_k = (booked["pred_rank"] <= k).sum()
    recall_at[k] = in_top_k / len(booked)
    print(f"Booking Recall@{k}: {recall_at[k]:.4f}  ({in_top_k:,}/{len(booked):,} booked hotels in top {k})")

# Click recall
clicked = val_feat[val_feat["click_bool"] == 1].copy()
click_recall5 = (clicked["pred_rank"] <= 5).sum() / len(clicked)
print(f"\nClick Recall@5: {click_recall5:.4f}")

# Mean booked rank
mean_booked_rank = booked["pred_rank"].mean()
median_booked_rank = booked["pred_rank"].median()
print(f"\nMean booked-hotel rank: {mean_booked_rank:.1f}")
print(f"Median booked-hotel rank: {median_booked_rank:.1f}")
print(f"% booked hotel at rank 1: {(booked['pred_rank'] == 1).mean():.2%}")

# Distribution of booked hotel rank
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(booked["pred_rank"], bins=range(1, int(booked["pred_rank"].max()) + 2),
        color="#4CAF50", edgecolor="black", linewidth=0.5, alpha=0.8)
ax.axvline(5.5, color="red", linestyle="--", linewidth=2, label="Top-5 cutoff")
ax.set_xlabel("Predicted Rank of Booked Hotel")
ax.set_ylabel("Count")
ax.set_title("Distribution of Booked Hotel Rank (lower = better)")
ax.legend()
plt.tight_layout()
plt.show()

# Performance on positive-only queries
pos_searches = set(val_feat[val_feat["click_bool"] == 1]["srch_id"].unique())
pos_val = val_feat[val_feat["srch_id"].isin(pos_searches)]
ndcg_pos = evaluate_ndcg(pos_val, score_col="pred_score", k=5)
print(f"\nNDCG@5 on positive-only queries (have click/booking): {ndcg_pos:.5f}")
print(f"NDCG@5 overall (includes zero-signal queries): {overall_ndcg:.5f}")

## 4.2 Winner-vs-Loser Within-Query Comparison

For searches with a booking, compare the booked hotel against the non-booked ones. What makes the winner win?

In [ ]:
# Winner vs loser analysis
booked_srch_ids = set(booked["srch_id"].unique())
val_booked_queries = val_feat[val_feat["srch_id"].isin(booked_srch_ids)].copy()

booked_rows = val_booked_queries[val_booked_queries["booking_bool"] == 1]
nonbooked_rows = val_booked_queries[val_booked_queries["booking_bool"] == 0]

compare_features = [
    "price_usd", "prop_starrating", "prop_review_score",
    "prop_location_score1", "prop_location_score2",
    "promotion_flag", "prop_brand_bool",
    "comp_rate_advantage", "orig_destination_distance",
]
compare_features = [f for f in compare_features if f in val_feat.columns]

print("=== Winner (Booked) vs Loser (Non-Booked) Within-Query ===")
print(f"{'Feature':<30} {'Booked':>12} {'Non-Booked':>12} {'Diff':>10} {'Effect':>10}")
print("-" * 80)
for feat in compare_features:
    bm = booked_rows[feat].mean()
    nm = nonbooked_rows[feat].mean()
    diff = bm - nm
    # Cohen's d
    pooled_std = np.sqrt((booked_rows[feat].var() + nonbooked_rows[feat].var()) / 2)
    d = diff / pooled_std if pooled_std > 0 else 0
    print(f"{feat:<30} {bm:>12.4f} {nm:>12.4f} {diff:>10.4f} {d:>10.3f}")

# Percentile analysis: where does the booked hotel rank within query on each feature?
print("\n=== Booked Hotel's Within-Query Percentile ===")
print("(Higher = booked hotel is better on this feature relative to alternatives)")
for feat in compare_features:
    if feat in val_booked_queries.columns:
        ascending = True if feat == "price_usd" else False
        val_booked_queries[f"_pct_{feat}"] = val_booked_queries.groupby("srch_id")[feat].rank(
            pct=True, ascending=not ascending)
        booked_pct = val_booked_queries.loc[val_booked_queries["booking_bool"]==1, f"_pct_{feat}"].mean()
        print(f"  {feat:<30} percentile={booked_pct:.3f}")

# Specific questions
print("\n=== How often is booked hotel... ===")
# Cheaper than median
g = val_booked_queries.groupby("srch_id")["price_usd"]
val_booked_queries["_q_median_price"] = g.transform("median")
cheaper = (booked_rows["price_usd"] <= booked_rows.index.map(
    lambda i: val_booked_queries.loc[i, "_q_median_price"] if i in val_booked_queries.index else np.nan
)).mean() if len(booked_rows) > 0 else 0
# Simpler approach
merged = booked_rows.merge(
    val_booked_queries.groupby("srch_id")["price_usd"].median().reset_index().rename(columns={"price_usd": "median_price"}),
    on="srch_id", how="left")
print(f"  Cheaper than query median: {(merged['price_usd'] <= merged['median_price']).mean():.2%}")

merged2 = booked_rows.merge(
    val_booked_queries.groupby("srch_id")["prop_starrating"].mean().reset_index().rename(columns={"prop_starrating": "mean_star"}),
    on="srch_id", how="left")
print(f"  Higher star than query mean: {(merged2['prop_starrating'] >= merged2['mean_star']).mean():.2%}")

print(f"  On promotion: {booked_rows['promotion_flag'].mean():.2%}")
if "comp_rate_advantage" in booked_rows.columns:
    print(f"  Positive competitor advantage: {(booked_rows['comp_rate_advantage'] > 0).mean():.2%}")

## 4.3 Hard-Negative Analysis

Compare the booked hotel vs the model's top-ranked non-booked hotel. This directly shows which features the model over/under-weights.

In [ ]:
# Hard negative: model's top predicted non-booked vs actual booked
hard_neg_data = []

for srch_id in booked_srch_ids:
    q = val_feat[val_feat["srch_id"] == srch_id]
    booked_row = q[q["booking_bool"] == 1].iloc[0] if len(q[q["booking_bool"] == 1]) > 0 else None
    if booked_row is None:
        continue
    # Model's top non-booked
    non_booked = q[q["booking_bool"] == 0].sort_values("pred_score", ascending=False)
    if len(non_booked) == 0:
        continue
    top_wrong = non_booked.iloc[0]
    
    row = {"srch_id": srch_id}
    for feat in compare_features:
        row[f"booked_{feat}"] = booked_row[feat]
        row[f"wrong_{feat}"] = top_wrong[feat]
        row[f"diff_{feat}"] = top_wrong[feat] - booked_row[feat]
    row["booked_rank"] = booked_row["pred_rank"]
    row["wrong_rank"] = top_wrong["pred_rank"]
    hard_neg_data.append(row)

hn_df = pd.DataFrame(hard_neg_data)
print(f"Hard-negative pairs: {len(hn_df):,}")

# Where does the model go wrong?
print("\n=== Hard Negative vs Booked: Mean Differences ===")
print("(Positive = model's wrong pick scores HIGHER on this feature)")
print(f"{'Feature':<30} {'Booked mean':>12} {'Wrong mean':>12} {'Wrong-Booked':>14} {'Pattern':>20}")
print("-" * 95)
for feat in compare_features:
    bm = hn_df[f"booked_{feat}"].mean()
    wm = hn_df[f"wrong_{feat}"].mean()
    diff = wm - bm
    pattern = ""
    if feat == "price_usd" and diff < 0:
        pattern = "model prefers cheaper"
    elif feat == "price_usd" and diff > 0:
        pattern = "model ok on price"
    elif diff > 0:
        pattern = "model over-weights"
    elif diff < 0:
        pattern = "model under-weights"
    print(f"{feat:<30} {bm:>12.4f} {wm:>12.4f} {diff:>14.4f} {pattern:>20}")

# Focus on failures (booked rank > 5)
failures = hn_df[hn_df["booked_rank"] > 5]
if len(failures) > 0:
    print(f"\n=== Same analysis for FAILURES ONLY (booked rank > 5, n={len(failures)}) ===")
    for feat in compare_features:
        bm = failures[f"booked_{feat}"].mean()
        wm = failures[f"wrong_{feat}"].mean()
        diff = wm - bm
        print(f"  {feat:<30} diff={diff:>+10.4f}")

## 4.4 "Easy Win Missed" Analysis

Failures where the booked hotel was obviously superior — these indicate broken feature interactions.

In [ ]:
# Easy wins missed: booked hotel was clearly better but model ranked it low
failures_df = val_feat[
    (val_feat["booking_bool"] == 1) & (val_feat["pred_rank"] > 5)
].copy()

if len(failures_df) > 0:
    # Merge with query stats
    q_stats = val_feat.groupby("srch_id").agg(
        median_price=("price_usd", "median"),
        mean_star=("prop_starrating", "mean"),
        mean_review=("prop_review_score", "mean"),
        mean_loc1=("prop_location_score1", "mean"),
    ).reset_index()
    
    failures_df = failures_df.merge(q_stats, on="srch_id", how="left")
    
    cheaper = failures_df["price_usd"] <= failures_df["median_price"]
    better_star = failures_df["prop_starrating"] >= failures_df["mean_star"]
    better_review = failures_df["prop_review_score"] >= failures_df["mean_review"]
    promoted = failures_df["promotion_flag"] == 1
    better_loc = failures_df["prop_location_score1"] >= failures_df["mean_loc1"]
    
    print(f"=== 'Easy Win Missed' — Booked hotel ranked > 5 ({len(failures_df):,} cases) ===")
    print(f"  Cheaper than median:          {cheaper.mean():.2%}")
    print(f"  Higher star than mean:         {better_star.mean():.2%}")
    print(f"  Higher review than mean:       {better_review.mean():.2%}")
    print(f"  On promotion:                  {promoted.mean():.2%}")
    print(f"  Better location than mean:     {better_loc.mean():.2%}")
    
    # Multi-criteria: how many dimensions was booked hotel superior?
    n_superior = cheaper.astype(int) + better_star.astype(int) + better_review.astype(int) + better_loc.astype(int)
    print(f"\n  Superior on 3+ dimensions:     {(n_superior >= 3).mean():.2%}  ({(n_superior >= 3).sum():,} cases)")
    print(f"  Superior on 4 dimensions:      {(n_superior >= 4).mean():.2%}  ({(n_superior >= 4).sum():,} cases)")
    
    if (n_superior >= 3).sum() > 0:
        print("\n  These 'easy' failures suggest the model is missing obvious quality signals.")
else:
    print("No failures to analyze (all booked hotels in top 5)")

## 4.5 Model Error Analysis — Where Does the Model Fail?

In [ ]:
# Error analysis by segment
booked_val = val_feat[val_feat["booking_bool"] == 1].copy()
success = booked_val[booked_val["pred_rank"] <= 5]
failure = booked_val[booked_val["pred_rank"] > 5]

print(f"=== Error Breakdown ===")
print(f"Booked hotels in top 5 (success): {len(success):,} ({len(success)/len(booked_val):.2%})")
print(f"Booked hotels outside top 5 (failure): {len(failure):,} ({len(failure)/len(booked_val):.2%})")
print(f"\nFailure rank distribution:")
for lo, hi, label in [(6,10,"6-10"), (11,20,"11-20"), (21,40,"21+")]:
    n = ((failure["pred_rank"] >= lo) & (failure["pred_rank"] <= hi)).sum()
    print(f"  Rank {label}: {n:,} ({n/len(failure):.1%})")

# Compare segments
print("\n=== Success vs Failure: Segment Comparison ===")
print(f"{'Segment':<35} {'Success rate':>14} {'N success':>12} {'N failure':>12}")
print("-" * 80)

# random_bool
for rv in [0, 1]:
    s = booked_val[booked_val["random_bool"] == rv]
    sr = (s["pred_rank"] <= 5).mean()
    ns = (s["pred_rank"] <= 5).sum()
    nf = (s["pred_rank"] > 5).sum()
    print(f"{'random_bool=' + str(rv):<35} {sr:>13.2%} {ns:>12,} {nf:>12,}")

# Domestic vs international
booked_val["_domestic"] = booked_val["visitor_location_country_id"] == booked_val["prop_country_id"]
for label, mask in [("Domestic", booked_val["_domestic"]), ("International", ~booked_val["_domestic"])]:
    s = booked_val[mask]
    if len(s) > 0:
        sr = (s["pred_rank"] <= 5).mean()
        print(f"{label:<35} {sr:>13.2%} {(s['pred_rank']<=5).sum():>12,} {(s['pred_rank']>5).sum():>12,}")

# Family vs no family
for label, mask in [("Family (children>0)", booked_val["srch_children_count"]>0),
                     ("No children", booked_val["srch_children_count"]==0)]:
    s = booked_val[mask]
    if len(s) > 0:
        sr = (s["pred_rank"] <= 5).mean()
        print(f"{label:<35} {sr:>13.2%} {(s['pred_rank']<=5).sum():>12,} {(s['pred_rank']>5).sum():>12,}")

# Cold-start: is booked property rare?
train_prop_counts = train_split.groupby("prop_id").size()
booked_val["_prop_count"] = booked_val["prop_id"].map(train_prop_counts).fillna(0)
for label, lo, hi in [("Cold-start (<5 impressions)", 0, 5), ("Known (5-100)", 5, 100), ("Popular (100+)", 100, 1e9)]:
    s = booked_val[(booked_val["_prop_count"] >= lo) & (booked_val["_prop_count"] < hi)]
    if len(s) > 0:
        sr = (s["pred_rank"] <= 5).mean()
        print(f"{label:<35} {sr:>13.2%} {(s['pred_rank']<=5).sum():>12,} {(s['pred_rank']>5).sum():>12,}")

# Group size
booked_val["_group_size"] = booked_val["srch_id"].map(val_feat.groupby("srch_id").size())
for label, lo, hi in [("Small group (<15)", 0, 15), ("Medium (15-30)", 15, 30), ("Large (30+)", 30, 100)]:
    s = booked_val[(booked_val["_group_size"] >= lo) & (booked_val["_group_size"] < hi)]
    if len(s) > 0:
        sr = (s["pred_rank"] <= 5).mean()
        print(f"{label:<35} {sr:>13.2%} {(s['pred_rank']<=5).sum():>12,} {(s['pred_rank']>5).sum():>12,}")

## 4.6 Query Difficulty Analysis

In [ ]:
# Query difficulty metrics
q_difficulty = val_feat.groupby("srch_id").agg(
    n_hotels=("prop_id", "count"),
    price_iqr=("price_usd", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    price_std=("price_usd", "std"),
    star_nunique=("prop_starrating", "nunique"),
    n_promoted=("promotion_flag", "sum"),
    n_branded=("prop_brand_bool", "sum"),
    has_booking=("booking_bool", "max"),
).reset_index()

# Compute NDCG per query
ndcg_per_q = []
for srch_id, group in val_feat.groupby("srch_id"):
    sorted_g = group.sort_values("pred_score", ascending=False)
    ndcg = ndcg_at_k(sorted_g["relevance"].values, k=5)
    ndcg_per_q.append({"srch_id": srch_id, "ndcg5": ndcg})
ndcg_q_df = pd.DataFrame(ndcg_per_q)
q_difficulty = q_difficulty.merge(ndcg_q_df, on="srch_id")

# Only positive queries
pos_q = q_difficulty[q_difficulty["has_booking"] == 1]

# Difficulty bins
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By group size
bins_gs = pd.qcut(pos_q["n_hotels"], q=4, duplicates="drop")
gs_ndcg = pos_q.groupby(bins_gs, observed=True)["ndcg5"].mean()
gs_ndcg.plot(kind="bar", ax=axes[0,0], color="#2196F3", edgecolor="black")
axes[0,0].set_title("NDCG@5 by Group Size")
axes[0,0].set_ylabel("Mean NDCG@5")
axes[0,0].tick_params(axis='x', rotation=45)

# By price spread
bins_ps = pd.qcut(pos_q["price_iqr"], q=4, duplicates="drop")
ps_ndcg = pos_q.groupby(bins_ps, observed=True)["ndcg5"].mean()
ps_ndcg.plot(kind="bar", ax=axes[0,1], color="#FF9800", edgecolor="black")
axes[0,1].set_title("NDCG@5 by Price IQR")
axes[0,1].tick_params(axis='x', rotation=45)

# By star variety
sv_ndcg = pos_q.groupby("star_nunique")["ndcg5"].mean()
sv_ndcg.plot(kind="bar", ax=axes[1,0], color="#4CAF50", edgecolor="black")
axes[1,0].set_title("NDCG@5 by Star Rating Variety")
axes[1,0].set_ylabel("Mean NDCG@5")

# By n_promoted
np_ndcg = pos_q.groupby(pos_q["n_promoted"].clip(upper=5))["ndcg5"].mean()
np_ndcg.plot(kind="bar", ax=axes[1,1], color="#9C27B0", edgecolor="black")
axes[1,1].set_title("NDCG@5 by # Promoted Hotels")
axes[1,1].tick_params(axis='x', rotation=0)

plt.suptitle("Query Difficulty Analysis", fontsize=14)
plt.tight_layout()
plt.show()

## 4.7 NDCG@5 by Segment

In [ ]:
# NDCG@5 by segment
def segment_ndcg(val_df, segment_col, segment_vals=None):
    results = []
    if segment_vals is None:
        segment_vals = val_df[segment_col].unique()
    for val in sorted(segment_vals):
        subset = val_df[val_df[segment_col] == val]
        if subset["srch_id"].nunique() < 10:
            continue
        ndcg = evaluate_ndcg(subset, score_col="pred_score", k=5)
        booked_sub = subset[subset["booking_bool"] == 1]
        mean_rank = booked_sub["pred_rank"].mean() if len(booked_sub) > 0 else np.nan
        results.append({
            "Segment": f"{segment_col}={val}",
            "N_searches": subset["srch_id"].nunique(),
            "NDCG@5": round(ndcg, 4),
            "Mean_booked_rank": round(mean_rank, 1) if not np.isnan(mean_rank) else "N/A",
        })
    return pd.DataFrame(results)

val_feat["_domestic"] = (val_feat["visitor_location_country_id"] == val_feat["prop_country_id"]).astype(int)
val_feat["_family"] = (val_feat["srch_children_count"] > 0).astype(int)
val_feat["_short_stay"] = (val_feat["srch_length_of_stay"] <= 2).astype(int)
val_feat["_last_minute"] = (val_feat["srch_booking_window"] < 3).astype(int)

all_segments = pd.concat([
    segment_ndcg(val_feat, "random_bool"),
    segment_ndcg(val_feat, "_domestic"),
    segment_ndcg(val_feat, "_family"),
    segment_ndcg(val_feat, "_short_stay"),
    segment_ndcg(val_feat, "_last_minute"),
])

print("=== NDCG@5 by Segment ===")
print(all_segments.to_string(index=False))

## 4.8 Popularity Bias (Model Side)

Use within-query rank (not raw scores) since LambdaRank scores are only comparable within the same query.

In [ ]:
# Popularity bias: model-side
train_prop_counts = train_split.groupby("prop_id").size()
val_feat["_pop"] = val_feat["prop_id"].map(train_prop_counts).fillna(0)

bins = [0, 5, 20, 50, 100, float("inf")]
labels_pop = ["1-5", "6-20", "21-50", "51-100", "100+"]
val_feat["_pop_bucket"] = pd.cut(val_feat["_pop"], bins=bins, labels=labels_pop)

pop_model = val_feat.groupby("_pop_bucket", observed=True).agg(
    mean_within_query_rank=("pred_rank", "mean"),
    mean_within_query_pct=("pred_rank_pct", "mean"),
    exposure_at_1=("pred_rank", lambda x: (x == 1).mean()),
    exposure_at_5=("pred_rank", lambda x: (x <= 5).mean()),
    n_rows=("pred_rank", "size"),
)

print("=== Model Popularity Bias (within-query metrics) ===")
print(pop_model.round(4).to_string())

# Booked hotel rank by popularity
booked_pop = val_feat[val_feat["booking_bool"] == 1].groupby("_pop_bucket", observed=True).agg(
    mean_booked_rank=("pred_rank", "mean"),
    booking_recall_5=("pred_rank", lambda x: (x <= 5).mean()),
    n_bookings=("pred_rank", "size"),
)
print("\n=== Booked Hotel Rank by Property Popularity ===")
print(booked_pop.round(4).to_string())

## 4.9 Fairness Metrics (Model Side)

For each bias dimension, compute group-level NDCG@5, recall@5, and mean booked rank.

In [ ]:
# Model-side fairness
print("=== Fairness Metrics (Model Side) ===\n")

fairness_dims = [
    ("Family", val_feat["srch_children_count"] > 0, val_feat["srch_children_count"] == 0, "Family", "No children"),
    ("Domestic", val_feat["_domestic"] == 1, val_feat["_domestic"] == 0, "Domestic", "International"),
    ("Brand", val_feat["prop_brand_bool"] == 1, val_feat["prop_brand_bool"] == 0, "Branded", "Independent"),
    ("Star tier", val_feat["prop_starrating"] >= 4, val_feat["prop_starrating"] <= 2, "High-star(4-5)", "Low-star(0-2)"),
]

print(f"{'Dimension':<15} {'Group A':<15} {'NDCG@5':>8} {'Recall@5':>10} {'MeanRank':>10}   {'Group B':<15} {'NDCG@5':>8} {'Recall@5':>10} {'MeanRank':>10}")
print("-" * 120)

for dim_name, mask_a, mask_b, label_a, label_b in fairness_dims:
    for label, mask in [(label_a, mask_a), (label_b, mask_b)]:
        subset = val_feat[mask]
        ndcg = evaluate_ndcg(subset, score_col="pred_score", k=5)
        booked_sub = subset[subset["booking_bool"] == 1]
        recall5 = (booked_sub["pred_rank"] <= 5).mean() if len(booked_sub) > 0 else 0
        mean_rank = booked_sub["pred_rank"].mean() if len(booked_sub) > 0 else 0
        if label == label_a:
            print(f"{dim_name:<15} {label:<15} {ndcg:>8.4f} {recall5:>10.4f} {mean_rank:>10.1f}", end="   ")
        else:
            print(f"{label:<15} {ndcg:>8.4f} {recall5:>10.4f} {mean_rank:>10.1f}")

## 4.10 Feature Importance

In [ ]:
# Feature importance
importance_gain = sorted(
    zip(feature_cols, model.feature_importance(importance_type="gain")),
    key=lambda x: x[1], reverse=True
)
importance_split = sorted(
    zip(feature_cols, model.feature_importance(importance_type="split")),
    key=lambda x: x[1], reverse=True
)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Gain
top_n = 25
feats_g = [x[0] for x in importance_gain[:top_n]]
vals_g = [x[1] for x in importance_gain[:top_n]]
axes[0].barh(range(top_n), vals_g[::-1], color="#2196F3", edgecolor="black", linewidth=0.5)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(feats_g[::-1], fontsize=9)
axes[0].set_title(f"Top {top_n} Features by Gain")

feats_s = [x[0] for x in importance_split[:top_n]]
vals_s = [x[1] for x in importance_split[:top_n]]
axes[1].barh(range(top_n), vals_s[::-1], color="#FF9800", edgecolor="black", linewidth=0.5)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(feats_s[::-1], fontsize=9)
axes[1].set_title(f"Top {top_n} Features by Split Count")

plt.suptitle("Feature Importance: Gain vs Split", fontsize=14)
plt.tight_layout()
plt.show()

## 4.11 Robustness: Multiple Validation Splits

In [ ]:
# Quick robustness check: 5 random seeds for train/val split
import gc

robustness_results = []
for seed in [42, 123, 456, 789, 2024]:
    tr, va = split_val(train_raw, val_frac=0.1, random_state=seed)
    tr = tr.sort_values("srch_id").reset_index(drop=True)
    va = va.sort_values("srch_id").reset_index(drop=True)
    
    tr_feat = build_features(tr, agg_source=tr, is_train=True)
    va_feat = build_features(va, agg_source=tr, is_train=False)
    fc = [c for c in feature_cols if c in va_feat.columns and c in tr_feat.columns]
    
    propensity = compute_position_propensity(train_raw)
    weights = compute_sample_weights(tr, propensity)
    
    tr_groups = tr_feat.groupby("srch_id").size().values
    va_groups = va_feat.groupby("srch_id").size().values
    
    dtrain = lgb.Dataset(tr_feat[fc], label=tr_feat["relevance"], group=tr_groups, weight=weights)
    dval = lgb.Dataset(va_feat[fc], label=va_feat["relevance"], group=va_groups, reference=dtrain)
    
    m = lgb.train(
        {"objective": "lambdarank", "metric": "ndcg", "eval_at": [5],
         "learning_rate": 0.03, "num_leaves": 400, "min_child_samples": 50,
         "subsample": 0.7, "colsample_bytree": 0.6, "reg_alpha": 0.1, "reg_lambda": 1.0,
         "seed": seed, "verbose": -1, "n_jobs": -1},
        dtrain, num_boost_round=1500, valid_sets=[dval],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    va_feat_eval = va_feat.copy()
    va_feat_eval["pred_score"] = m.predict(va_feat[fc])
    ndcg = evaluate_ndcg(va_feat_eval, score_col="pred_score", k=5)
    robustness_results.append({"seed": seed, "best_iter": m.best_iteration, "ndcg5": ndcg})
    print(f"  Seed {seed}: best_iter={m.best_iteration}, NDCG@5={ndcg:.5f}")
    
    del tr, va, tr_feat, va_feat, va_feat_eval, m
    gc.collect()

rob_df = pd.DataFrame(robustness_results)
print(f"\nMean NDCG@5: {rob_df['ndcg5'].mean():.5f}")
print(f"Std NDCG@5:  {rob_df['ndcg5'].std():.5f}")
print(f"Min:         {rob_df['ndcg5'].min():.5f}")
print(f"Max:         {rob_df['ndcg5'].max():.5f}")

## 4.12 Prediction Score Analysis

In [ ]:
# Score distribution by relevance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
val_feat.boxplot(column="pred_score", by="relevance", ax=axes[0])
axes[0].set_title("Predicted Score by True Relevance")
axes[0].set_xlabel("Relevance (0=ignored, 1=clicked, 5=booked)")
axes[0].set_ylabel("Predicted Score")
plt.sca(axes[0])
plt.title("Predicted Score by True Relevance")

# Overlapping histograms
for rel, color, label in [(0, "#2196F3", "Ignored (0)"), (1, "#FF9800", "Clicked (1)"), (5, "#4CAF50", "Booked (5)")]:
    subset = val_feat[val_feat["relevance"] == rel]["pred_score"]
    axes[1].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
axes[1].set_xlabel("Predicted Score")
axes[1].set_ylabel("Density")
axes[1].set_title("Score Distribution by Relevance")
axes[1].legend()

plt.tight_layout()
plt.show()

# Score separation stats
for rel in [0, 1, 5]:
    s = val_feat[val_feat["relevance"] == rel]["pred_score"]
    print(f"Relevance {rel}: mean={s.mean():.4f}, median={s.median():.4f}, std={s.std():.4f}")

---

## Final Summary — Actionable Findings

Key diagnostics to review:
1. **Recall@K**: What fraction of booked hotels end up in top K?
2. **Winner analysis**: What features make booked hotels win?
3. **Hard negatives**: What does the model incorrectly prefer? (over/under-weighted features)
4. **Easy wins missed**: Are obvious winners being missed?
5. **Segment failures**: Which segments have systematically worse performance?
6. **Robustness**: Is NDCG@5 stable across different validation splits?
7. **Fairness**: Are there group-level performance disparities to address?

Use these findings to iterate on feature engineering and model configuration.